![Week 3 Day 1](../../assets/banners/w3d1.png)

# Week 3 Day 1 - Solution: Redesigning a Bad Tool

The exercise: turn `def marks(x, y): return QUESTIONS[x].criteria[y].marks` into a tool a model can't misuse, then run it through the hand-made ReAct loop.

⏮ Back to the lab: `week3/day1.ipynb`

💡 **Fast & cheap run.** Two lines in `.env` decide what this notebook costs. `MODEL_MODE=free` runs on OpenRouter's free models for $0 (same `OPENROUTER_API_KEY`, no credits, but only about 50 requests a day), and `DATA_SIZE=small` uses a small slice of the data (2 PDFs · Q1–Q3 · 2 students) when you just want to see the APIs work. Restart the kernel after editing `.env`; the next cell prints what you're running.

In [ ]:
import json
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

from gradewise import datasets
from gradewise.config import DATA_DIR, settings
from gradewise.schemas import ExamPaper

load_dotenv(override=True)

# 🔑 cheap and free both use one key, OPENROUTER_API_KEY (from openrouter.ai/keys); custom may need others
for key_name in settings.required_keys:
    api_key = os.getenv(key_name)
    if api_key:
        print(f"✅ {key_name} exists and begins {api_key[:8]}")
    else:
        print(f"ℹ️ {key_name} not set. That's OK: only the cells marked 🔑 need a key, the rest run offline.")
        print("   To set one up, see Part 5 of the setup guide (and remember to SAVE .env!)")

print(settings.describe_models())
print(datasets.describe())

exam = ExamPaper.model_validate_json(
    (DATA_DIR / "exams" / "cs_midterm" / "expected_exam.json").read_text(encoding="utf-8")
)
QUESTIONS = {q.id: q for q in exam.questions}

### The redesigned tool
What changed, and why:
- **Name** says exactly what it returns: `get_criterion_marks`, not `marks`
- **Docstring** says when to use it, with example IDs; `parse_docstring=True` puts the argument descriptions in the schema
- **Arguments** are named and typed. The criterion is looked up by its **ID** (`"Q3.c2"`), not a list position the model would have to guess
- **Errors** come back as helpful text listing the valid IDs, so the model can correct itself
- **Output** is one short line, not a JSON dump

In [ ]:
@tool(parse_docstring=True)
def get_criterion_marks(question_id: str, criterion_id: str) -> str:
    """Get how many marks one marking criterion of a CS201 midterm question is worth, and what it rewards.

    Use this when you need the marks for a single criterion. For a whole question's rubric, use get_rubric.

    Args:
        question_id: The question ID, e.g. "Q3" or "Q2a".
        criterion_id: The criterion ID, which starts with the question ID, e.g. "Q3.c2".
    """
    qid = question_id.strip()
    q = QUESTIONS.get(qid[:1].upper() + qid[1:])  # 💡 forgiving about "q3" vs "Q3"
    if q is None:
        return f"No question {question_id!r}. Valid IDs: {', '.join(QUESTIONS)}"
    criterion = next((c for c in q.criteria if c.id.lower() == criterion_id.strip().lower()), None)
    if criterion is None:
        return f"{q.id} has no criterion {criterion_id!r}. Its criteria are: {', '.join(c.id for c in q.criteria)}"
    return f"{criterion.id} is worth {criterion.marks:g} of {q.id}'s {q.max_marks:g} marks: {criterion.description}"


print(json.dumps(convert_to_openai_tool(get_criterion_marks)["function"], indent=2))
for args in [
    {"question_id": "Q3", "criterion_id": "Q3.c2"},
    {"question_id": "Q30", "criterion_id": "x"},
    {"question_id": "Q3", "criterion_id": "Q3.c9"},
]:
    print("→", get_criterion_marks.invoke(args))

💡 Note the small kindness of upper-casing the first letter: a model that writes `"q3"` still gets an answer. Be strict about *meaning*, forgiving about *formatting*.

### Through the ReAct loop, with a stand-in model
The stand-in first makes a **mistake** (a criterion that doesn't exist), reads the helpful error, and corrects itself: exactly the behaviour good error messages make possible.

In [ ]:
# 🎭 A STAND-IN model, so you can watch agents work with no API key and no cost.
# 💡 GenericFakeChatModel replies with the messages we script, in order. It can't really call
#    tools, so bind_tools() just returns the same model: the tool calls are in our script.
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage


class ScriptedModel(GenericFakeChatModel):
    """Replays a script of AIMessages. ⚠️ A teaching prop, not an LLM: it never reads your prompt."""

    def bind_tools(self, tools, **kwargs):
        return self


def scripted(*replies: AIMessage) -> ScriptedModel:
    return ScriptedModel(messages=iter(replies))


def call(name: str, call_id: str = "call_1", **args) -> AIMessage:
    """A scripted AIMessage that asks for ONE tool call, exactly like a real model would."""
    return AIMessage(content="", tool_calls=[{"name": name, "args": args, "id": call_id}])

In [ ]:
tools = {get_criterion_marks.name: get_criterion_marks}


def react(model, question: str, max_steps: int = 5) -> str:
    messages = [
        SystemMessage("You are GradeWise, a CS201 marking assistant. Use tools; never guess marks."),
        HumanMessage(question),
    ]
    for step in range(1, max_steps + 1):
        ai = model.invoke(messages)
        messages.append(ai)
        if not ai.tool_calls:
            return ai.text
        for tc in ai.tool_calls:
            observation = tools[tc["name"]].invoke(tc["args"])
            print(f"Step {step} 🛠️  {tc['name']}({tc['args']})\n        👀 {observation}")
            messages.append(ToolMessage(observation, tool_call_id=tc["id"]))
    return "Stopped: too many steps."


standin = scripted(
    call("get_criterion_marks", question_id="Q3", criterion_id="Q3.c4"),  # ⚠️ a mistake: Q3 has 3 criteria
    call("get_criterion_marks", call_id="call_2", question_id="Q3", criterion_id="Q3.c3"),
    AIMessage(content="Naming a second, different deadlock condition (Q3.c3) is worth 1 mark."),
)
print("💬", react(standin, "How many marks for naming the second deadlock condition in Q3?"))

### 🔑 With a real model

In [ ]:
from gradewise.models import chat_model

print(
    "💬",
    react(
        chat_model(temperature=0).bind_tools([get_criterion_marks]), "How many marks is the last criterion of Q5 worth?"
    ),
)